In [49]:
import fitz
import re
from pathlib import Path
import random

NUM_LINE_RE = re.compile(
    r"^\s*[\(\[\{]?\s*[₹$€£]?\s*[-+]?"
    r"\d[\d,]*(?:\.\d+)?%?"
    r"\s*[\)\]\}]?\s*$"
)

In [60]:
def generate_probes(w, n_probes=25):
    left = w * 0.15
    right = w * 0.85

    rng = random.Random(42) # deterministic

    probes = sorted(
        rng.uniform(left, right)
        for _ in range(n_probes)
    )

    return probes

def longest_consecutive_run(hits, threshold=1):
    longest = 0
    current = 0

    for h in hits:
        if h > threshold:
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest

def extract_numeric_lines(page):
    numeric_lines = []
    page_dict = page.get_text("dict")
    margin_x = page.rect.width * 0.15
    margin_y = page.rect.height * 0.10

    work_rect = fitz.Rect(
        margin_x,
        margin_y,
        page.rect.width - margin_x,
        page.rect.height - margin_y
    )
        
    
    for block in page_dict["blocks"]:

        if block["type"] != 0:
            continue
        for line in block["lines"]:
            
            bbox = fitz.Rect(line["bbox"])

            if not work_rect.intersects(bbox):
                continue
            
            text = "".join(span["text"] for span in line["spans"]).strip()
            if NUM_LINE_RE.match(text):

                numeric_lines.append({
                    "text": text,
                    "bbox": bbox,
                    "cx": (bbox.x0 + bbox.x1) / 2
                })

    return numeric_lines

def probe_numeric_columns(page, numeric_lines, n_probes=40):

    w = page.rect.width
    probes = generate_probes(w)

    hits = [0] * n_probes
    for item in numeric_lines:
        x0 = item["bbox"].x0
        x1 = item["bbox"].x1
        for i, px in enumerate(probes):

            if x0 <= px <= x1:
                hits[i] += 1

    return probes, hits

def debug_numeric_lines(page, numeric_lines, probes):

    for item in numeric_lines:

        page.draw_rect(
            item["bbox"],
            color=(1,0,0),
            width=0.8,
            overlay=True
        )

    for px in probes:

        page.draw_line(
            fitz.Point(px,0),
            fitz.Point(px,page.rect.height),
            color=(0,0,1),
            width=0.5,
            overlay=True
        )

In [61]:
pdf_path = r"C:\Users\rando\Office Projects\rep_fetchapi\PDF_SAMPLES\annual\2025-ADANIENT.pdf"
pdf_file = Path(pdf_path)
doc = fitz.open(pdf_path)
probe_data = []

for page_no in range(doc.page_count):

    page = doc[page_no]

    numeric_lines = extract_numeric_lines(page)
    probes, hits = probe_numeric_columns(page, numeric_lines)

    row = {
        "pdf": pdf_file.stem,
        "page": page_no + 1,
        "numeric_lines": len(numeric_lines),
        # "matched_lines": "|".join(line["text"] for line in numeric_lines),
        "max_hits": max(hits),
        "total_hits":sum(hits),
        "hits_1":len([i for i in hits if i >1]),
        "cons_hit": longest_consecutive_run(hits, threshold=2),
        # "avg_hits": round(sum(hits) / len(hits), 2)
    }
    for i, h in enumerate(hits):
        row[f"probe_{i+1}"] = h


    probe_data.append(row)
    
doc.close()

In [62]:
import pandas as pd

df = pd.DataFrame(probe_data)
df.to_excel("probe_hits.xlsx", index=False)

In [24]:
from collections import Counter

def top_two_modes(values):
    c = Counter(values)
    return c.most_common(2)

hits = top_two_modes(df["max_hits"])
# [(0, 82), (1, 34)]

lines = top_two_modes(df["numeric_lines"])
# [(0, 81), (1, 36)]

print(f"HITS: {hits}, LINES: {lines}")

HITS: [(1, 296), (3, 27)], LINES: [(1, 160), (2, 85)]


In [42]:
max_modes = {int(m) for m, _ in top_two_modes(df["max_hits"])}
num_modes = {int(m) for m, _ in top_two_modes(df["numeric_lines"])}

max_modes.add(0)
num_modes.add(0)

print(max_modes)
print(num_modes)

filtered = df[
    ~(
        df["max_hits"].astype(int).isin(max_modes) &
        df["numeric_lines"].astype(int).isin(num_modes)
    )
].copy()

{0, 1, 3}
{0, 1, 2}


In [44]:
filtered.to_excel("FINAL_PDF.xlsx", index = False)